# 05 · Flatten the dome with the built-in map projections

Four projections live in `mt.geodesicdome.projection`: `KavrayskiyVII`, `WagnerVI`, `WagnerIII` and
`EqualEarth`. `projection.build(dome)` projects every vertex (stored in `vertex.projected_coord`) and returns
an (M, 3) array of triangle indices. Triangles that would wrap around the ±180° meridian are dropped, so
M is a little smaller than 20 *f*². The projections treat +z as the north pole (latitude = arcsin *z*).

To *rotate* the sphere inside a projection, without losing triangles at the seam, see notebook 09.

Script version: [`examples/05_map_projections.py`](../05_map_projections.py) · Needs: numpy, matplotlib, ipywidgets (optional: `ipympl` for live, rotatable figures)

<sub>SPDX-License-Identifier: AGPL-3.0-or-later · Copyright (C) 2022-2026 Masahiro Takatsuka. See the NOTICE file for attribution terms.</sub>

## Set-up

In [ ]:
import nb_setup  # makes mt.geodesicdome and examples/dome_utils.py importable from a checkout

WIDGET = nb_setup.setup()  # live figures with ipympl, static images otherwise

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from ipywidgets import Checkbox, Dropdown, IntSlider, interact
from matplotlib.collections import PolyCollection
from nb_setup import LiveFigure

from mt.geodesicdome.grid.geodesicdome import GeodesicDome
from mt.geodesicdome.projection.equal_earth import EqualEarth
from mt.geodesicdome.projection.kavrayskiy import KavrayskiyVII
from mt.geodesicdome.projection.wagner import WagnerIII, WagnerVI

PROJECTIONS = {'Kavrayskiy VII': KavrayskiyVII, 'Wagner VI': WagnerVI, 'Wagner III': WagnerIII,
               'Equal Earth': EqualEarth}

## Project and draw

In [ ]:
def draw_projection(ax, dome, projection, colouring='icosahedron face', edges=True):
    xyz = dome.get_all_xyz()
    triangles = projection.build(dome)                                    # (M, 3) vertex ids
    flat = np.array([v.projected_coord for v in dome.get_all_vertices()])  # (N, 2), row = vertex.id
    centres = xyz[triangles].mean(axis=1)

    if colouring == 'icosahedron face':
        ico = GeodesicDome(1)
        ico_centres = ico.get_all_xyz()[ico.get_all_triangles().reshape(-1, 3)].mean(axis=1)
        colours = plt.cm.tab20(np.argmax(centres @ ico_centres.T, axis=1) % 20)
    elif colouring == 'position (xyz as RGB)':
        c = centres / np.linalg.norm(centres, axis=1, keepdims=True)
        colours = 0.35 + 0.6 * (0.5 + 0.5 * c)
    else:                                                                   # latitude
        lat = np.arcsin(np.clip(centres[:, 2] / np.linalg.norm(centres, axis=1), -1, 1))
        colours = plt.cm.coolwarm(0.5 + lat / np.pi)

    ax.add_collection(PolyCollection(flat[triangles], facecolors=colours,
                                     edgecolors='white' if edges else 'none', linewidths=0.3))
    ax.autoscale()
    ax.set_aspect('equal')
    ax.set_axis_off()
    f = dome.frequency
    ax.set_title(f'{type(projection).__name__}  ({len(triangles)} of {20 * f ** 2} triangles)')

## All four projections, coloured by parent icosahedron face

In [ ]:
FREQ = 8
dome = GeodesicDome(FREQ)

fig, axes = plt.subplots(2, 2, figsize=(14, 7.5))
for ax, cls in zip(axes.flat, PROJECTIONS.values(), strict=False):
    draw_projection(ax, dome, cls())
fig.suptitle(f'GeodesicDome(frequency={FREQ}) in four map projections, '
             'coloured by parent icosahedron face', fontsize=14)
fig.tight_layout()
plt.show()

### Try it

In [ ]:
live = LiveFigure(figsize=(10, 5.5))
domes = {}


@interact(name=Dropdown(options=list(PROJECTIONS), value='Equal Earth', description='projection'),
          f=IntSlider(value=8, min=1, max=24, description='frequency'),
          colouring=Dropdown(options=['icosahedron face', 'position (xyz as RGB)', 'latitude'],
                             description='colour by'),
          edges=Checkbox(value=True, description='edges'))
def explore(name, f, colouring, edges):
    if f not in domes:
        domes[f] = GeodesicDome(f)
    fig = live.clear()
    draw_projection(fig.add_subplot(), domes[f], PROJECTIONS[name](), colouring, edges)
    fig.tight_layout()
    live.refresh()


live.show()